## Baseline - модель

## 1. Загрузка данных

In [ ]:
import sys
sys.path.append("..")

from src.data_loader import load_pkl, load_or_build_cache
from src.features import build_feature_cache

pkl_data = load_pkl()
label_cache = load_or_build_cache()
feature_cache = build_feature_cache(pkl_data, label_cache, "../data/raw/mosei.hdf5")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import os

os.makedirs("../figures/architectures", exist_ok=True)

fig, ax = plt.subplots(figsize=(7, 8.5))
ax.set_xlim(0, 10)
ax.set_ylim(0, 12.2)
ax.axis("off")

def box(x, y, w, h, title, subtitle, color, edge):
    rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.08",
                                     linewidth=1.3, edgecolor=edge, facecolor=color)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h*0.64, title, ha="center", va="center", fontsize=11, fontweight="bold")
    ax.text(x + w/2, y + h*0.28, subtitle, ha="center", va="center", fontsize=9, color="#333")

def arrow(x1, y1, x2, y2, color="#5F5E5A"):
    a = FancyArrowPatch((x1, y1), (x2, y2), arrowstyle="-|>", mutation_scale=14,
                          linewidth=1.2, color=color)
    ax.add_patch(a)

GRAY_FILL, GRAY_EDGE = "#F1EFE8", "#5F5E5A"
PURPLE_FILL, PURPLE_EDGE = "#EEEDFE", "#534AB7"
TEAL_FILL, TEAL_EDGE = "#E1F5EE", "#0F6E56"

# Общий ствол
box(2, 10.2, 6, 1.3, "Входной вектор признаков", "размерность зависит от признака (300/2000/76/72)", GRAY_FILL, GRAY_EDGE)
arrow(5, 10.2, 5, 9.7)
box(2, 8.4, 6, 1.3, "Полносвязный слой 1", "256, ReLU, Dropout 0.3", GRAY_FILL, GRAY_EDGE)
arrow(5, 8.4, 5, 7.9)
box(2, 6.6, 6, 1.3, "Полносвязный слой 2", "128, ReLU, Dropout 0.3", GRAY_FILL, GRAY_EDGE)

# Ветвление на 2 головы
arrow(5, 6.6, 2.8, 5.3, color=PURPLE_EDGE)
arrow(5, 6.6, 7.2, 5.3, color=TEAL_EDGE)

box(1, 4.0, 3.6, 1.3, "Sentiment head", "Linear 128 → 3 класса", PURPLE_FILL, PURPLE_EDGE)
box(5.4, 4.0, 3.6, 1.3, "Emotion head", "Linear 128 → 6 меток", TEAL_FILL, TEAL_EDGE)

arrow(2.8, 4.0, 2.8, 2.9, color=PURPLE_EDGE)
arrow(7.2, 4.0, 7.2, 2.9, color=TEAL_EDGE)

box(1, 1.6, 3.6, 1.3, "CrossEntropyLoss", "3 класса sentiment", PURPLE_FILL, PURPLE_EDGE)
box(5.4, 1.6, 3.6, 1.3, "BCEWithLogitsLoss", "6 меток, с маской", TEAL_FILL, TEAL_EDGE)

plt.tight_layout()
plt.savefig("../figures/architectures/baseline_architecture.png", dpi=200, bbox_inches="tight")
plt.show()

## 2. Проверка работы моделей

In [ ]:
import torch
from src.models import MultitaskFFN

for name, dim in [("text_glove", 300), ("text_tfidf", 2000),
                   ("audio.mfcc", 76), ("audio.prosody", 72)]:
    model = MultitaskFFN(input_dim=dim)
    dummy = torch.randn(8, dim)
    sent_out, emo_out = model(dummy)
    print(f"{name:15s} | input={dim:5d} | sentiment_out={tuple(sent_out.shape)} "
          f"| emotion_out={tuple(emo_out.shape)} | params={model.count_parameters():,}")

In [ ]:
import numpy as np
from src.metrics import sentiment_metrics, emotion_metrics, print_report

np.random.seed(42)
N = 200

y_true_sent = np.random.randint(0, 3, size=N)
y_pred_sent = np.random.randint(0, 3, size=N)

y_true_emo = np.random.randint(0, 2, size=(N, 6))
y_pred_proba = np.random.rand(N, 6)

sm = sentiment_metrics(y_true_sent, y_pred_sent)
em = emotion_metrics(y_true_emo, y_pred_proba)
print_report("synthetic (random)", sm, em)

y_true_emo_edge = y_true_emo.copy()
y_true_emo_edge[:, 5] = 0
em_edge = emotion_metrics(y_true_emo_edge, y_pred_proba)
print("\nfear AUC при отсутствии позитивных случаев:", em_edge["per_emotion_auc"]["fear"])
print("Упало без ошибки:", "да" if True else "нет")

In [ ]:
from torch.utils.data import DataLoader
from src.models import MultitaskFFN
from src.train import (MOSEIFeatureDataset, compute_sentiment_class_weights,
                        compute_emotion_pos_weights, fit)

X_train = feature_cache["text_glove"]["train"]
X_valid = feature_cache["text_glove"]["valid"]

train_ds = MOSEIFeatureDataset(
    X_train, label_cache["train"]["sentiment_class"],
    label_cache["train"]["emotion_binary"], label_cache["train"]["mask"],
)
valid_ds = MOSEIFeatureDataset(
    X_valid, label_cache["valid"]["sentiment_class"],
    label_cache["valid"]["emotion_binary"], label_cache["valid"]["mask"],
)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=64, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Устройство:", device)

sent_weights = compute_sentiment_class_weights(label_cache["train"]["sentiment_class"])
emo_weights = compute_emotion_pos_weights(label_cache["train"]["emotion_binary"], label_cache["train"]["mask"])
print("Веса классов sentiment:", sent_weights)
print("pos_weight emotion:", emo_weights)

model = MultitaskFFN(input_dim=X_train.shape[1])
model, history = fit(model, train_loader, valid_loader, device,
                      sent_weights, emo_weights, epochs=5, patience=3)

## 3. Обучение baseline

In [ ]:
from src.train import (MOSEIFeatureDataset, compute_sentiment_class_weights,
                        compute_emotion_pos_weights, fit, evaluate)

In [ ]:
import pandas as pd

FEATURE_CONFIGS = [
    ("text", "GloVe (mean-pool)", "text_glove"),
    ("text", "TF-IDF", "text_tfidf"),
    ("audio", "MFCC (mean+std)", "audio.mfcc"),
    ("audio", "Prosody+VQ (mean+std)", "audio.prosody"),
]

def get_feature_array(feature_cache, key, split):
    if "." in key:
        main, sub = key.split(".")
        return feature_cache[main][split][sub]
    return feature_cache[key][split]


results = []
histories = {}
trained_models = {}

for modality, feat_name, feat_key in FEATURE_CONFIGS:
    print(f"\n{'='*60}\n{modality} / {feat_name}\n{'='*60}")

    X_train = get_feature_array(feature_cache, feat_key, "train")
    X_valid = get_feature_array(feature_cache, feat_key, "valid")
    X_test = get_feature_array(feature_cache, feat_key, "test")

    train_ds = MOSEIFeatureDataset(X_train, label_cache["train"]["sentiment_class"],
                                    label_cache["train"]["emotion_binary"], label_cache["train"]["mask"])
    valid_ds = MOSEIFeatureDataset(X_valid, label_cache["valid"]["sentiment_class"],
                                    label_cache["valid"]["emotion_binary"], label_cache["valid"]["mask"])
    test_ds = MOSEIFeatureDataset(X_test, label_cache["test"]["sentiment_class"],
                                   label_cache["test"]["emotion_binary"], label_cache["test"]["mask"])

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
    valid_loader = DataLoader(valid_ds, batch_size=64, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

    sent_weights = compute_sentiment_class_weights(label_cache["train"]["sentiment_class"])
    emo_weights = compute_emotion_pos_weights(label_cache["train"]["emotion_binary"], label_cache["train"]["mask"])

    model = MultitaskFFN(input_dim=X_train.shape[1])
    n_params = model.count_parameters()

    model, history = fit(model, train_loader, valid_loader, device,
                          sent_weights, emo_weights, epochs=30, patience=5, verbose=False)

    sm_test, em_test = evaluate(model, test_loader, device)

    results.append({
        "modality": modality,
        "feature": feat_name,
        "input_dim": X_train.shape[1],
        "n_params": n_params,
        "epochs_trained": len(history),
        "sentiment_macro_f1": round(sm_test["macro_f1_3class"], 3),
        "sentiment_acc2": round(sm_test["acc2"], 3),
        "emotion_macro_f1": round(em_test["macro_f1"], 3),
        "emotion_mean_auc": round(em_test["mean_auc"], 3),
    })
    histories[feat_key] = history
    trained_models[feat_key] = model

    print(f"TEST: sentiment macro-F1={sm_test['macro_f1_3class']:.3f} | "
          f"emotion macro-F1={em_test['macro_f1']:.3f} | epochs={len(history)}")

results_df = pd.DataFrame(results)
results_df.to_csv("../experiments/results.csv", index=False)
results_df

In [ ]:
import os
os.makedirs("../experiments", exist_ok=True)

results_df.to_csv("../experiments/results.csv", index=False)
results_df

## 4. Построение графиков

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for feat_key, hist in histories.items():
    epochs = [h["epoch"] for h in hist]
    val_f1 = [h["val_sentiment_macro_f1"] for h in hist]
    axes[0].plot(epochs, val_f1, marker="o", markersize=3, label=feat_key)

axes[0].set_xlabel("эпоха")
axes[0].set_ylabel("val sentiment macro-F1")
axes[0].set_title("Динамика sentiment macro-F1 на validation")
axes[0].legend()

for feat_key, hist in histories.items():
    epochs = [h["epoch"] for h in hist]
    train_loss = [h["train_loss_sent"] + h["train_loss_emo"] for h in hist]
    axes[1].plot(epochs, train_loss, marker="o", markersize=3, label=feat_key)

axes[1].set_xlabel("эпоха")
axes[1].set_ylabel("train loss (sentiment + emotion)")
axes[1].set_title("Динамика суммарного train loss")
axes[1].legend()

plt.tight_layout()
plt.savefig("../figures/architectures/baseline_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
import os
os.makedirs("../figures/architectures", exist_ok=True)

plt.savefig("../figures/architectures/baseline_training_curves.png", dpi=150, bbox_inches="tight")

Построены 4 baseline-конфигурации (2 типа признаков × 2 модальности: GloVe и TF-IDF для текста, MFCC и Prosody+VQ для аудио), каждая - feed-forward сеть с общим стволом (2 скрытых слоя, 256→128, ReLU, dropout 0.3) и двумя задачными головами (sentiment: 3 класса, emotion: 6 независимых бинарных меток). Обучение проводилось с весами классов, компенсирующими дисбаланс, выявленный в EDA (раздел 3.3), и маскированием эмоциональной головы на сэмплах без надёжной эмоциональной метки (раздел 3.2).
Текстовая модальность превосходит акустическую по обеим задачам: лучший результат на тестовой выборке достигнут конфигурацией GloVe (sentiment macro-F1 = 0,589, Acc2 = 0,794; emotion macro-F1 = 0,409, mean-AUC = 0,685). Среди текстовых признаков GloVe превосходит TF-IDF (0,589 против 0,553 по sentiment), что объясняется ограниченным объёмом обучающей выборки (16 265 сэмплов), недостаточным для эффективного обучения частотных представлений TF-IDF (2000 признаков, 546 тыс. параметров модели), тогда как GloVe использует предобученные на больших корпусах эмбеддинги.
Среди акустических признаков просодика и индикаторы качества голоса (Prosody+VQ) превосходят MFCC по обеим задачам (sentiment macro-F1: 0,444 против 0,407; emotion macro-F1: 0,377 против 0,358), что указывает на больший вклад характеристик интонации и тона голоса в выражение эмоций и сентимента по сравнению со спектральной огибающей речи в данном корпусе.
Количество эпох до срабатывания early stopping варьируется в зависимости от ёмкости модели: конфигурация TF-IDF (546 тыс. параметров) останавливается на 6-й эпохе, что свидетельствует о склонности к переобучению при высокой размерности признакового пространства относительно объёма выборки, тогда как компактные конфигурации (MFCC, GloVe) обучаются 16–17 эпох.
Полученные результаты определяют ожидания для следующего раздела: усложнение текстового энкодера (RNN/Transformer над полной последовательностью GloVe-векторов, а не только усреднённым представлением) представляется наиболее перспективным направлением улучшения, наряду с объединением модальностей, способным компенсировать ограничения каждой отдельной модальности.